In [101]:
import numpy as np
import matplotlib.pyplot as plt
import iadpython as iad
import empylib.waveoptics as wv

In [102]:
"""Simulate R/T for two scattering layers with different host indices."""
# Optical properties for layer 1
mu_s1 = 10.0      # scattering coefficient [1/mm]
mu_a1 = 0.1       # absorption coefficient [1/mm]
g1 = 0.9          # anisotropy
d1 = 1.0          # thickness [mm]
n1 = 1.4          # refractive index of layer 1

# Optical properties for layer 2
mu_s2 = 5.0       # scattering coefficient [1/mm]
mu_a2 = 0.05      # absorption coefficient [1/mm]
g2 = 0.8          # anisotropy
d2 = 1.5          # thickness [mm]
n2 = 1.6          # refractive index of layer 2

In [103]:
# Convert to single–scattering albedo and optical thickness
a1 = mu_s1 / (mu_a1 + mu_s1)
b1 = (mu_a1 + mu_s1) * d1
# a2 = mu_s2 / (mu_a2 + mu_s2)
# b2 = (mu_a2 + mu_s2) * d2
a2 = a1
b2 = b1
g2 = g1
n2 = n1

In [104]:
def interface_rt_matrix(sample: iad.Sample, n_i: float, n_t: float) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Construct diagonal reflection/transmission matrices for an index interface.

    Given an ``iad.Sample`` (which defines the quadrature angles and
    weights via ``sample.nu`` and ``sample.twonuw``) and two
    refractive indices, return the reflection and transmission
    operators suitable for use in :func:`iadpython.combine.add_layers`.

    Each operator is returned as a diagonal matrix with entries
    ``r_i / (2*nu_i*w_i)`` or ``t_i / (2*nu_i*w_i)``, where ``r_i``
    is the unpolarized Fresnel reflection coefficient for light
    incident from ``n_i`` onto ``n_t`` at the cosine angle ``nu_i``,
    and ``t_i = 1 - r_i``.  This scaling matches the conventions
    described by Prahl *et al.*【499810635147300†L280-L307】 and those
    implemented in :func:`iadpython.start.boundary_matrices`.

    Parameters
    ----------
    sample : iadpython.Sample
        Reference sample that defines the quadrature angles and
        weights.  Only ``sample.nu`` and ``sample.twonuw`` are used.
    n_i : float
        Refractive index of the incident medium.
    n_t : float
        Refractive index of the transmitted medium.

    Returns
    -------
    R_dn, R_up, T_dn, T_up : ndarray
        Diagonal matrices representing the reflection and transmission
        operators for radiation incident from the top (``dn``) and
        from the bottom (``up``).  For a simple interface the
        operators are the same in both directions, but they are
        returned separately for clarity.
    """
    # Fresnel reflection for unpolarized light at each quadrature angle
    r = iad.fresnel_reflection(n_i, sample.nu, n_t)
    t = 1.0 - r
    # Normalize by 2*nu*w; this matches the representation used by boundary_matrices
    diag_scale = sample.twonuw
    # Avoid division by zero for directions with nu=0; sample.twonuw has zero at nu=0
    with np.errstate(divide="ignore", invalid="ignore"):
        r_norm = np.divide(r, diag_scale, out=np.zeros_like(r), where=diag_scale != 0)
        t_norm = np.divide(t, diag_scale, out=np.zeros_like(t), where=diag_scale != 0)
    R = np.diagflat(r_norm)
    T = np.diagflat(t_norm)
    return R, R, T, T

In [105]:
# Create a reference sample for layer 1; this defines the quadrature
sample1 = iad.Sample(a=a1, b=b1, g=g1, n=n1, 
                          n_above = 1.0, n_below = n2, quad_pts=8)

# Compute intrinsic R/T matrices for layer 1
R1, T1 = iad.simple_layer_matrices(sample1)

# Compute intrinsic R/T matrices for layer 2 on the basis of layer 1
# We keep the quadrature and host index of the first layer to
# maintain compatible dimensions for star multiplication.
sample2 = iad.Sample(a=a2, b=b2, g=g2, n=n2, 
                          n_above = n1, n_below = 1.0, quad_pts=8)

R2, T2 = iad.simple_layer_matrices(sample2)

# Top boundary: air → layer 1
R01_int, R10_int, T01_int, T10_int = iad.boundary_matrices(sample1, top=True)

# mid boundary: layer 1 → layer 2
R12_int, R21_int, T12_int, T21_int = iad.boundary_matrices(sample2, top=True)

# Bottom boundary: layer 2 → air
R23_int, R32_int, T23_int, T32_int = iad.boundary_matrices(sample2, top=False)

# Add top boundary to layer 1
R01, R10, T01, T10 = iad.add_layers(sample1, R01_int, R10_int, T01_int, T10_int, R1, R1, T1, T1)

# Add internal interface between layer 1 and layer 2
R02, R20, T02, T20 = iad.add_layers(sample1, R01, R10, T01, T10, R12_int, R21_int, T12_int, T21_int)

# Add layer 2
R02, R20, T02, T20 = iad.add_layers(sample1, R02, R20, T02, T20, R2, R2, T2, T2)

# Add bottom boundary (layer 2 → air)
R03, R30, T03, T30 = iad.add_layers(sample1, R02, R20, T02, T20, R23_int, R32_int, T23_int, T32_int)

# Convert final matrices into total reflection/transmission
ur1, ut1, uru, utu = sample1.UX1_and_UXU(R03, T03)

print(f"ur1: {ur1: .5f}, ut1: {ut1: .5f}, uru: {uru: .5f}, utu: {utu: .5f}")

ur1:  0.28069, ut1:  0.27023, uru:  0.32571, utu:  0.22873


In [106]:
a = [a1, a2]
b = [b1, b2]
g = [g1, g2]
s = iad.Sample(a = a, b = b, g = g, n = n1, quad_pts = 8)

R12, T12           = iad.simple_layer_matrices(s)
# Reflectanve and transmittance of boundaries
R01, R10, T01, T10 = iad.boundary_matrices(s, top=True)   # top boundary
R23, R32, T23, T32 = iad.boundary_matrices(s, top=False)  # bottom boundary

# Use transfer matrix method to get total R and T
R02, R20, T02, T20 = iad.add_layers(s, R01, R10, T01, T10, R12, R12, T12, T12) 
R03, R30, T03, T30 = iad.add_layers(s, R02, R20, T02, T20, R23, R32, T23, T32)

ur1, ut1, uru, utu = s.UX1_and_UXU(R03,T03)
print(f"ur1: {ur1: .5f}, ut1: {ut1: .5f}, uru: {uru: .5f}, utu: {utu: .5f}")

ur1:  0.24101, ut1:  0.30991, uru:  0.27990, utu:  0.27454


In [107]:
s = iad.Sample(a = a1, b = 2*b1, g = g1, n = n1, quad_pts = 8)

ur1, ut1, uru, utu = s.rt()
print(f"ur1: {ur1: .5f}, ut1: {ut1: .5f}, uru: {uru: .5f}, utu: {utu: .5f}")

ur1:  0.24101, ut1:  0.30991, uru:  0.27990, utu:  0.27454
